# Circuits — the ones built for telecommunications

Everything so far has been general. This notebook is about the specific jobs a radio has to do: select one frequency out of many, hand power to an antenna without wasting it, carry a signal down a cable, and get information back off a carrier.

Only two ideas are genuinely new, and both come from the previous notebook. The first is that **impedance depends on frequency**, which is all a filter is. The second is that **impedance is what a source sees**, which is all matching is.

$$H(j\omega)\ \text{shapes what passes}
\qquad\qquad
Z_0=\sqrt{L/C}\ \text{decides what reflects}$$

The last two sections break the linearity that has held throughout the series. A diode used as a detector and a multiplier used as a mixer both create frequencies that were not in their inputs — which is impossible for any linear circuit, and is precisely why every radio contains at least one nonlinear part.

The schematics animate as before, and the transmission line is simulated as an actual ladder of inductors and capacitors so you can watch a pulse travel and come back.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.collections import LineCollection
import ipywidgets as widgets
from IPython.display import display

BG, PANEL, FG = "#05070b", "#0a0d14", "#c9cfda"
MUTED, GRIDC = "#6b7280", "#1b2130"
POS, NEG, DOT = "#3fd0c9", "#e0555c", "#ffd24a"
BLUE, ORANGE, GREEN, PURP = "#5aa9e6", "#e08a3c", "#7ddc7d", "#b48ce0"
VMAP = mpl.colors.LinearSegmentedColormap.from_list(
    "volt", [(0.0, NEG), (0.5, "#4a5060"), (1.0, POS)])

plt.rcParams.update({
    "figure.dpi": 112, "font.size": 8.5, "axes.titlesize": 9,
    "figure.facecolor": BG, "savefig.facecolor": BG, "axes.facecolor": PANEL,
    "axes.edgecolor": GRIDC, "axes.labelcolor": FG, "text.color": FG,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": GRIDC,
    "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.facecolor": PANEL, "legend.edgecolor": GRIDC, "legend.framealpha": 0.9,
})
SL = {"style": {"description_width": "104px"},
      "layout": widgets.Layout(width="290px"), "continuous_update": False}


def panel(ax, edge=None, lw=1.3):
    ax.set_facecolor(PANEL)
    for s in ax.spines.values():
        s.set_visible(True); s.set_color(edge or GRIDC)
        s.set_linewidth(lw if edge else 0.8)
    ax.tick_params(colors=MUTED, labelsize=7)
    return ax


def readout(fig, x, y, lines, color=FG, size=7.4):
    fig.text(x, y, "\n".join(lines), family="monospace", fontsize=size,
             color=color, va="top", ha="left", linespacing=1.55)


def footer(fig, text):
    fig.text(0.010, 0.012, text, family="monospace", fontsize=6.6, color=MUTED)
    fig.text(0.990, 0.012, "circuits · telecom", family="monospace",
             fontsize=6.6, color=MUTED, ha="right")


def timeline(n, step=1, interval=90, desc="time"):
    p = widgets.Play(value=0, min=0, max=n, step=step, interval=interval)
    s = widgets.IntSlider(value=0, min=0, max=n, step=step, description=desc + ":",
                          continuous_update=False,
                          style={"description_width": "104px"},
                          layout=widgets.Layout(width="430px"))
    widgets.jslink((p, "value"), (s, "value"))
    return p, s


# ----- schematic primitives, Falstad style -------------------------------
def vcolor(v, vmax):
    return VMAP(np.clip(0.5 + 0.5 * v / max(vmax, 1e-9), 0, 1))


def wire(ax, pts, v, vmax, lw=2.6):
    pts = np.asarray(pts, float)
    seg = np.stack([pts[:-1], pts[1:]], axis=1)
    ax.add_collection(LineCollection(seg, colors=[vcolor(v, vmax)] * len(seg),
                                     linewidths=lw, zorder=2))


def node_dot(ax, p, v, vmax, s=34):
    ax.plot(*p, "o", ms=np.sqrt(s), color=vcolor(v, vmax), zorder=4)


def resistor(ax, p0, p1, v, vmax, label=None, n=6, amp=0.16):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    a, b = p0 + u * L * 0.28, p1 - u * L * 0.28
    ts = np.linspace(0, 1, 2 * n + 1)
    zz = [a + (b - a) * t + nrm * amp * ((-1) ** k if 0 < k < 2 * n else 0)
          for k, t in enumerate(ts)]
    wire(ax, [p0, a], v, vmax)
    wire(ax, zz, v, vmax, lw=2.2)
    wire(ax, [b, p1], v, vmax)
    if label:
        ax.text(*(0.5 * (p0 + p1) + nrm * 0.34), label, color=FG, fontsize=7.5,
                ha="center", va="center")


def capacitor(ax, p0, p1, v, vmax, label=None, gap=0.10, half=0.24):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    c = 0.5 * (p0 + p1)
    a, b = c - u * gap, c + u * gap
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    for q in (a, b):
        ax.plot(*np.stack([q - nrm * half, q + nrm * half]).T, color=FG, lw=2.4,
                zorder=3)
    if label:
        ax.text(*(c + nrm * 0.40), label, color=FG, fontsize=7.5, ha="center")


def inductor(ax, p0, p1, v, vmax, label=None, coils=4, r=0.13):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    a, b = p0 + u * L * 0.25, p1 - u * L * 0.25
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    seg = np.linalg.norm(b - a) / coils
    for k in range(coils):
        c = a + u * seg * (k + 0.5)
        th = np.linspace(0, np.pi, 24)
        pts = np.array([c + u * (seg / 2) * np.cos(np.pi - t) + nrm * r * np.sin(t)
                        for t in th])
        ax.plot(pts[:, 0], pts[:, 1], color=FG, lw=2.0, zorder=3)
    if label:
        ax.text(*(0.5 * (p0 + p1) + nrm * 0.38), label, color=FG, fontsize=7.5,
                ha="center")


def diode(ax, p0, p1, v, vmax, label=None, s=0.20):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    c = 0.5 * (p0 + p1)
    a, b = c - u * s, c + u * s
    wire(ax, [p0, a], v, vmax); wire(ax, [b, p1], v, vmax)
    ax.add_patch(mpatches.Polygon([a + nrm * s, a - nrm * s, b], closed=True,
                                  facecolor=ORANGE, edgecolor=ORANGE, zorder=3))
    ax.plot(*np.stack([b - nrm * s, b + nrm * s]).T, color=FG, lw=2.6, zorder=3)
    if label:
        ax.text(*(c + nrm * 0.40), label, color=FG, fontsize=7.5, ha="center")


def source(ax, p0, p1, v, vmax, kind="dc", label=None, r=0.30):
    p0, p1 = np.asarray(p0, float), np.asarray(p1, float)
    c = 0.5 * (p0 + p1)
    d = p1 - p0; L = max(np.linalg.norm(d), 1e-9); u = d / L
    nrm = np.array([-u[1], u[0]])
    wire(ax, [p0, c - u * r], v, vmax); wire(ax, [c + u * r, p1], v, vmax)
    ax.add_patch(mpatches.Circle(c, r, fill=False, ec=FG, lw=2.0, zorder=3))
    if kind == "dc":
        ax.plot(*np.stack([c - u * 0.12 - nrm * 0.16, c - u * 0.12 + nrm * 0.16]).T,
                color=FG, lw=2.6, zorder=4)
        ax.plot(*np.stack([c + u * 0.12 - nrm * 0.09, c + u * 0.12 + nrm * 0.09]).T,
                color=FG, lw=2.0, zorder=4)
    else:
        t = np.linspace(-1, 1, 40)
        pts = np.array([c + u * (0.19 * t[i]) + nrm * 0.15 * np.sin(np.pi * t[i])
                        for i in range(len(t))])
        ax.plot(pts[:, 0], pts[:, 1], color=FG, lw=1.8, zorder=4)
    if label:
        ax.text(*(c + nrm * (r + 0.22)), label, color=FG, fontsize=7.5, ha="center")


def path_len(pts):
    p = np.asarray(pts, float)
    d = np.linalg.norm(np.diff(p, axis=0), axis=1)
    return np.r_[0, np.cumsum(d)]


def charge_dots(ax, loop, q, spacing=0.42, ms=4.2):
    """Yellow dots at arclength q + n*spacing — this is the current, visualised."""
    p = np.asarray(loop, float)
    s = path_len(p)
    L = s[-1]
    if L <= 0:
        return
    offs = (np.arange(0, L, spacing) + (q % spacing)) % L
    x = np.interp(offs, s, p[:, 0]); y = np.interp(offs, s, p[:, 1])
    ax.plot(x, y, "o", ms=ms, color=DOT, zorder=5, mec="none")


def sch_axes(ax, xlim, ylim):
    panel(ax)
    ax.set_xlim(*xlim); ax.set_ylim(*ylim)
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    return ax


def loop_rect(x0, x1, y0, y1, n=60):
    top = np.stack([np.linspace(x0, x1, n), np.full(n, y1)], 1)
    right = np.stack([np.full(n, x1), np.linspace(y1, y0, n)], 1)
    bot = np.stack([np.linspace(x1, x0, n), np.full(n, y0)], 1)
    left = np.stack([np.full(n, x0), np.linspace(y0, y1, n)], 1)
    return np.vstack([top, right, bot, left])


def seg(p0, p1, n=40):
    return np.stack([np.linspace(p0[0], p1[0], n),
                     np.linspace(p0[1], p1[1], n)], 1)


def g_butter(n):
    """Normalised Butterworth ladder prototype values."""
    return np.array([2 * np.sin((2 * k - 1) * np.pi / (2 * n))
                     for k in range(1, n + 1)])


def phasor(ax, z, color, label=None, lw=2.2, tail=(0, 0)):
    ax.annotate("", xy=(tail[0] + z.real, tail[1] + z.imag), xytext=tail,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=lw))
    if label:
        ax.text(tail[0] + z.real * 1.12, tail[1] + z.imag * 1.12, label,
                color=color, fontsize=8, ha="center", va="center")


def cplane(ax, lim):
    panel(ax)
    ax.axhline(0, color=GRIDC, lw=0.9); ax.axvline(0, color=GRIDC, lw=0.9)
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    return ax


print("engine ready — telecom notebook")
print(f"Butterworth n=3 prototype: {np.round(g_butter(3), 4)}  (should be 1 2 1)")

engine ready — telecom notebook
Butterworth n=3 prototype: [1. 2. 1.]  (should be 1 2 1)


## Selectivity — what more poles buy

A single RC gives 20 dB per decade, which is nowhere near enough to separate one radio channel from its neighbour. Cascading $n$ poles gives $20n$, and the standard way to place them is the Butterworth response:

$$|H(j\omega)|^2=\frac{1}{1+(f/f_c)^{2n}}$$

Every order passes through exactly $-3.0103$ dB at $f_c$ — verified for $n=1,2,3,5$ — and the far skirt falls at exactly $20n$ dB/decade: $-20$, $-40$, $-60$, $-100$ dB at ten times the corner. The passband is as flat as it can be made, which is why this shape is the default when nothing else is specified.

Flatness is a choice, though, and it is not free. Allowing a little ripple in the passband buys a much steeper skirt for the same number of components — the Chebyshev response. The panel puts both on the same axes at the same order so the trade is visible: same parts, more selectivity, at the cost of a passband that is no longer flat and a phase response that is worse.

Phase is the quantity nobody notices until it matters. Each pole eventually contributes 90°, so a 5-pole filter swings 450° across its band, and different frequencies come out at different delays. For voice that is invisible; for data it is intersymbol interference.

In [2]:
def butter_H(f, fc, n):
    return 1 / np.sqrt(1 + (f / fc) ** (2 * n))


def cheby_H(f, fc, n, ripple_db):
    eps = np.sqrt(10 ** (ripple_db / 10) - 1)
    x = f / fc
    Tn = np.where(x <= 1, np.cos(n * np.arccos(np.clip(x, -1, 1))),
                  np.cosh(n * np.arccosh(np.maximum(x, 1.0))))
    return 1 / np.sqrt(1 + (eps * Tn) ** 2)


def draw_order(n, ripple_db, fc_khz, f_khz):
    fc = fc_khz * 1e3
    ff = np.logspace(np.log10(fc / 100), np.log10(fc * 100), 900)
    Hb = butter_H(ff, fc, n)
    Hc = cheby_H(ff, fc, n, ripple_db)
    fnow = f_khz * 1e3

    fig = plt.figure(figsize=(13.0, 4.9))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.25, 0.55],
                          wspace=0.28, hspace=0.44, left=0.055, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = panel(fig.add_subplot(gs[0, 0]), BLUE)
    for k in (1, 2, 3, 5):
        a0.semilogx(ff / 1e3, 20 * np.log10(butter_H(ff, fc, k)),
                    color=POS if k == n else MUTED, lw=1.9 if k == n else 0.9,
                    alpha=1.0 if k == n else 0.55)
        a0.text(ff[-1] / 1e3, 20 * np.log10(butter_H(ff[-1], fc, k)),
                f" n={k}", color=POS if k == n else MUTED, fontsize=7,
                va="center")
    a0.axvline(fc / 1e3, color=DOT, lw=1.0)
    a0.axhline(-3.01, color=MUTED, lw=0.8, ls=":")
    a0.set_ylim(-90, 8); a0.set_ylabel("|H|  (dB)")
    a0.set_title(f"Butterworth — every order is −3.01 dB at fc, "
                 f"skirt = 20n dB/decade")

    a1 = panel(fig.add_subplot(gs[1, 0]), ORANGE)
    a1.semilogx(ff / 1e3, 20 * np.log10(Hb), color=POS, lw=1.7,
                label=f"Butterworth n={n}")
    a1.semilogx(ff / 1e3, 20 * np.log10(Hc), color=ORANGE, lw=1.7,
                label=f"Chebyshev {ripple_db:.1f} dB ripple")
    a1.axvline(fc / 1e3, color=DOT, lw=1.0)
    a1.axhline(-ripple_db, color=MUTED, lw=0.7, ls=":")
    a1.set_xlim(fc / 1e3 / 8, fc / 1e3 * 8); a1.set_ylim(-70, 6)
    a1.set_xlabel("frequency  (kHz)"); a1.set_ylabel("|H|  (dB)")
    a1.legend(fontsize=7)
    a1.set_title("same order, more selectivity, ripple as the price")

    a2 = panel(fig.add_subplot(gs[:, 1]), PURP)
    ph = np.zeros_like(ff)
    for k in range(n):
        pole = np.exp(1j * np.pi * (2 * k + 1 + n) / (2 * n))
        ph += np.degrees(np.angle(1 / (1j * ff / fc - pole)))
    a2.semilogx(ff / 1e3, ph, color=PURP, lw=1.7)
    a2.axvline(fc / 1e3, color=DOT, lw=1.0)
    a2.axvline(f_khz, color=FG, lw=1.0, ls="--")
    for m in range(1, n + 1):
        a2.axhline(-90 * m, color=GRIDC, lw=0.6)
    a2.set_xlabel("frequency  (kHz)"); a2.set_ylabel("phase  (deg)")
    a2.set_title(f"phase — each pole eventually contributes 90°, "
                 f"total {90*n}°")

    rej = 20 * np.log10(butter_H(np.array([fc * 10]), fc, n))[0]
    readout(fig, 0.845, 0.90, [
        "FILTER", "─" * 26,
        f"order n     {n:>10d}",
        f"fc          {fc_khz:>10.2f}kHz",
        f"poles       {n:>10d}",
        f"skirt       {20*n:>10d}dB/dec",
        "", "BUTTERWORTH", "─" * 26,
        f"at fc       {20*np.log10(butter_H(np.array([fc]),fc,n))[0]:>+10.4f}dB",
        f"at 2fc      {20*np.log10(butter_H(np.array([2*fc]),fc,n))[0]:>+10.3f}dB",
        f"at 10fc     {rej:>+10.3f}dB",
        f"phase swing {90*n:>10d}°",
        "", "CHEBYSHEV", "─" * 26,
        f"ripple      {ripple_db:>10.2f}dB",
        f"at 2fc      {20*np.log10(cheby_H(np.array([2*fc]),fc,n,ripple_db))[0]:>+10.3f}dB",
        f"advantage   "
        f"{20*np.log10(cheby_H(np.array([2*fc]),fc,n,ripple_db))[0]-20*np.log10(butter_H(np.array([2*fc]),fc,n))[0]:>+10.2f}dB",
        "", "AT THIS f", "─" * 26,
        f"f           {f_khz:>10.2f}kHz",
        f"f/fc        {fnow/fc:>10.4f}",
        f"|H| butter  {20*np.log10(butter_H(np.array([fnow]),fc,n))[0]:>+10.3f}dB",
    ])
    footer(fig, f"|H|² = 1/(1+(f/fc)^2n)   ·   n = {n}   ·   "
                f"−3.0103 dB at fc for every order   ·   skirt {20*n} dB/decade")
    plt.show()


w1 = dict(n=widgets.IntSlider(value=3, min=1, max=7, step=1,
                              description="order n:", **SL),
          ripple_db=widgets.FloatSlider(value=0.5, min=0.05, max=3.0, step=0.05,
                                        description="ripple (dB):", **SL),
          fc_khz=widgets.FloatSlider(value=100, min=10, max=1000, step=10,
                                     description="fc (kHz):", **SL),
          f_khz=widgets.FloatSlider(value=100, min=1, max=2000, step=1,
                                    description="marker f (kHz):", **SL))
display(widgets.HBox([w1["n"], w1["ripple_db"], w1["fc_khz"], w1["f_khz"]]),
        widgets.interactive_output(draw_order, w1))

Output()

## The LC ladder — turning a response into component values

A Butterworth curve is a specification. Turning it into parts uses tabulated normalised values $g_k$ for a 1 Ω, 1 rad/s prototype, then scaling both axes:

$$L_k=\frac{g_kR_0}{2\pi f_c},\qquad
C_k=\frac{g_k}{R_0\,2\pi f_c}$$

The prototype numbers are exact and worth recognising: $n=3$ gives $1,\,2,\,1$; $n=5$ gives $0.618,\,1.618,\,2,\,1.618,\,0.618$ — those are golden-ratio numbers, because the poles sit on a circle. The panel prints them and the denormalised values in henries and farads.

The ladder alternates series inductors and shunt capacitors for a reason you can read off the previous notebook: a series inductor's impedance rises with frequency and a shunt capacitor's falls, so both push high frequencies away from the output. Each additional rung is one more pole.

Note the terminations. The design assumes a specific source and load resistance and the response is only correct when both are present — an LC filter is not a component you can drop anywhere. Change the load and the passband ripples, which is the panel's other curve and the reason filters and matching are always designed together.

In [3]:
def ladder_response(f, Ls, Cs, Rs, RL):
    """Cascade ABCD matrices: series L, shunt C, alternating."""
    s = 2j * np.pi * np.atleast_1d(f)
    M = np.array([[np.ones_like(s), np.zeros_like(s)],
                  [np.zeros_like(s), np.ones_like(s)]])
    def mul(M, N):
        return np.array([[M[0, 0] * N[0, 0] + M[0, 1] * N[1, 0],
                          M[0, 0] * N[0, 1] + M[0, 1] * N[1, 1]],
                         [M[1, 0] * N[0, 0] + M[1, 1] * N[1, 0],
                          M[1, 0] * N[0, 1] + M[1, 1] * N[1, 1]]])
    li, ci = 0, 0
    for k in range(len(Ls) + len(Cs)):
        if k % 2 == 0:
            Z = s * Ls[li]; li += 1
            N = np.array([[np.ones_like(s), Z],
                          [np.zeros_like(s), np.ones_like(s)]])
        else:
            Y = s * Cs[ci]; ci += 1
            N = np.array([[np.ones_like(s), np.zeros_like(s)],
                          [Y, np.ones_like(s)]])
        M = mul(M, N)
    Vr = RL / (M[0, 0] * RL + M[0, 1] + Rs * (M[1, 0] * RL + M[1, 1]))
    return Vr * (Rs + RL) / RL          # normalised to the matched flat value


def draw_ladder(n, fc_khz, R0, RL, f_khz):
    fc = fc_khz * 1e3
    w = 2 * np.pi * fc
    g = g_butter(n)
    Ls, Cs = [], []
    for k in range(n):
        if k % 2 == 0:
            Ls.append(g[k] * R0 / w)
        else:
            Cs.append(g[k] / (R0 * w))
    ff = np.logspace(np.log10(fc / 50), np.log10(fc * 50), 700)
    Hm = np.abs(ladder_response(ff, Ls, Cs, R0, RL))
    Hm = Hm / np.max(Hm[ff < fc / 20]) if np.any(ff < fc / 20) else Hm
    Hmatch = np.abs(ladder_response(ff, Ls, Cs, R0, R0))
    Hmatch = Hmatch / np.max(Hmatch[ff < fc / 20])
    vmax = 1.0

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.3, 1.15, 0.55],
                          wspace=0.28, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[0, 0])
    sch_axes(a0, (-0.5, 6.2), (-0.6, 2.2))
    x = 0.9
    wire(a0, [(0.3, 1.7), (x, 1.7)], 1.0, vmax)
    source(a0, (0.3, 0.2), (0.3, 1.7), 0.5, vmax, "ac", None)
    li = ci = 0
    for k in range(n):
        if k % 2 == 0:
            inductor(a0, (x, 1.7), (x + 1.1, 1.7), 0.8 - 0.1 * k, vmax,
                     f"L{li+1}")
            x += 1.1; li += 1
        else:
            capacitor(a0, (x, 1.7), (x, 0.2), 0.5, vmax, f"C{ci+1}")
            ci += 1
    wire(a0, [(x, 1.7), (x + 0.7, 1.7)], 0.3, vmax)
    resistor(a0, (x + 0.7, 1.7), (x + 0.7, 0.2), 0.2, vmax, f"{RL:.0f}Ω")
    wire(a0, [(0.3, 0.2), (x + 0.7, 0.2)], 0.0, vmax)
    a0.set_title(f"order {n} ladder — series L, shunt C, alternating",
                 fontsize=8.5)

    a1 = panel(fig.add_subplot(gs[1, 0]), GREEN)
    a1.bar(range(n), g, color=[PURP if k % 2 == 0 else ORANGE for k in range(n)],
           width=0.6)
    for k, gg in enumerate(g):
        a1.text(k, gg, f"{gg:.4f}", ha="center", va="bottom", fontsize=7)
    a1.set_xticks(range(n))
    a1.set_xticklabels([f"g{k+1}" for k in range(n)], fontsize=7.5)
    a1.set_title("normalised prototype values", fontsize=8.5)

    a2 = panel(fig.add_subplot(gs[:, 1]), BLUE)
    a2.semilogx(ff / 1e3, 20 * np.log10(Hmatch), color=POS, lw=1.8,
                label=f"matched ({R0:.0f} Ω)")
    a2.semilogx(ff / 1e3, 20 * np.log10(Hm), color=ORANGE, lw=1.4,
                label=f"load {RL:.0f} Ω")
    a2.semilogx(ff / 1e3, 20 * np.log10(butter_H(ff, fc, n)), color=MUTED,
                lw=0.9, ls="--", label="ideal Butterworth")
    a2.axvline(fc / 1e3, color=DOT, lw=1.0)
    a2.axvline(f_khz, color=FG, lw=1.0, ls="--")
    a2.axhline(-3.01, color=MUTED, lw=0.7, ls=":")
    a2.set_ylim(-80, 8); a2.set_xlabel("frequency  (kHz)")
    a2.set_ylabel("|H|  (dB)"); a2.legend(fontsize=7)
    a2.set_title("the design is only correct at its intended terminations")

    lines = ["DESIGN", "─" * 26,
             f"order       {n:>10d}",
             f"fc          {fc_khz:>10.2f}kHz",
             f"R0 design   {R0:>10.0f}Ω",
             f"RL actual   {RL:>10.0f}Ω", "", "COMPONENTS", "─" * 26]
    li = ci = 0
    for k in range(n):
        if k % 2 == 0:
            lines.append(f"L{li+1}          {Ls[li]*1e6:>10.4f}µH"); li += 1
        else:
            lines.append(f"C{ci+1}          {Cs[ci]*1e9:>10.4f}nF"); ci += 1
    im = np.argmin(np.abs(ff - fc))
    lines += ["", "RESPONSE", "─" * 26,
              f"at fc       {20*np.log10(Hmatch[im]):>+10.3f}dB",
              f"ideal       {-3.0103:>+10.3f}dB",
              f"mismatch    {20*np.log10(Hm[im])-20*np.log10(Hmatch[im]):>+10.3f}dB"]
    readout(fig, 0.845, 0.90, lines, size=7.0)
    footer(fig, f"L = g·R0/ω_c    C = g/(R0·ω_c)   ·   "
                f"prototype {np.round(g,4)}")
    plt.show()


w2 = dict(n=widgets.IntSlider(value=3, min=2, max=7, step=1,
                              description="order n:", **SL),
          fc_khz=widgets.FloatSlider(value=1000, min=100, max=5000, step=50,
                                     description="fc (kHz):", **SL),
          R0=widgets.FloatSlider(value=50, min=25, max=300, step=5,
                                 description="design R0 (Ω):", **SL),
          RL=widgets.FloatSlider(value=50, min=10, max=600, step=10,
                                 description="actual load (Ω):", **SL),
          f_khz=widgets.FloatSlider(value=1000, min=50, max=8000, step=50,
                                    description="marker f (kHz):", **SL))
display(widgets.VBox([widgets.HBox([w2["n"], w2["fc_khz"], w2["R0"]]),
                      widgets.HBox([w2["RL"], w2["f_khz"]])]),
        widgets.interactive_output(draw_ladder, w2))

Output()

## Impedance matching — making a load look like something else

Maximum power transfer said the load should equal the source resistance. Antennas and amplifiers rarely oblige, and you cannot simply add a resistor to make up the difference — that would burn the very power you are trying to deliver. **Reactances** can transform a resistance without dissipating anything.

The L-network is the minimum solution: one series and one shunt reactance.

$$Q=\sqrt{\frac{R_{high}}{R_{low}}-1},\qquad
X_s=QR_{low},\qquad
X_p=\frac{R_{high}}{Q}$$

The panel computes the input impedance of the matched network and reports $|Z_{in}-R_{low}|$ at exactly $0.00\times10^{0}$ Ω, with $|\Gamma|=0$ and VSWR $=1.000000$. The match is not approximate; at the design frequency it is exact.

What it is not is **broadband**. $Q$ is fixed by the transformation ratio alone, so a bigger ratio forces a higher $Q$ and therefore a narrower match — the bandwidth curve narrows visibly as you separate the two resistances. Matching 50 Ω to 300 Ω needs $Q=2.236$; matching 50 Ω to 5000 Ω needs $Q=9.95$ and a band a quarter as wide. When one L-network is too narrow, the answer is two in cascade through an intermediate resistance.

In [4]:
def lmatch(Rl, Rh, f0, config):
    Q = np.sqrt(max(Rh / Rl - 1, 1e-12))
    Xs, Xp = Q * Rl, Rh / Q
    w0 = 2 * np.pi * f0
    if config == "series L, shunt C":
        Ls, Cp = Xs / w0, 1 / (w0 * Xp)
        return Q, Xs, Xp, Ls, Cp
    Cs, Lp = 1 / (w0 * Xs), Xp / w0
    return Q, Xs, Xp, Cs, Lp


def zin_match(f, Rl, Rh, f0, config):
    w = 2 * np.pi * np.atleast_1d(f)
    Q, Xs, Xp, a, b = lmatch(Rl, Rh, f0, config)
    if config == "series L, shunt C":
        Zs = 1j * w * a
        Yp = 1j * w * b
    else:
        Zs = 1 / (1j * w * a)
        Yp = 1 / (1j * w * b)
    Zp = 1 / (Yp + 1 / Rh)
    return Zs + Zp


def draw_match(Rl, Rh, f0_mhz, config, f_mhz):
    f0 = f0_mhz * 1e6
    Q, Xs, Xp, a, b = lmatch(Rl, Rh, f0, config)
    Zin0 = zin_match(f0, Rl, Rh, f0, config)[0]
    ff = np.linspace(f0 * 0.2, f0 * 2.2, 900)
    Z = zin_match(ff, Rl, Rh, f0, config)
    G = np.abs((Z - Rl) / (Z + Rl))
    VSWR = (1 + G) / np.maximum(1 - G, 1e-12)
    Znow = zin_match(f_mhz * 1e6, Rl, Rh, f0, config)[0]
    Gnow = abs((Znow - Rl) / (Znow + Rl))
    vmax = 1.0

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.2, 1.25, 0.55],
                          wspace=0.28, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[0, 0])
    sch_axes(a0, (-0.5, 5.2), (-0.5, 2.2))
    wire(a0, [(0.3, 1.7), (1.2, 1.7)], 1.0, vmax)
    source(a0, (0.3, 0.2), (0.3, 1.7), 0.5, vmax, "ac", f"{Rl:.0f}Ω src")
    if config == "series L, shunt C":
        inductor(a0, (1.2, 1.7), (2.6, 1.7), 0.7, vmax, f"L {a*1e9:.1f}nH")
        capacitor(a0, (3.2, 1.7), (3.2, 0.2), 0.4, vmax, f"C {b*1e12:.1f}pF")
    else:
        capacitor(a0, (1.2, 1.7), (2.6, 1.7), 0.7, vmax, f"C {a*1e12:.1f}pF")
        inductor(a0, (3.2, 1.7), (3.2, 0.2), 0.4, vmax, f"L {b*1e9:.1f}nH")
    wire(a0, [(2.6, 1.7), (4.4, 1.7)], 0.4, vmax)
    resistor(a0, (4.4, 1.7), (4.4, 0.2), 0.2, vmax, f"{Rh:.0f}Ω load")
    wire(a0, [(0.3, 0.2), (4.4, 0.2)], 0.0, vmax)
    node_dot(a0, (1.2, 1.7), 1.0, vmax)
    a0.text(1.2, 1.95, "sees 50 Ω", color=DOT, fontsize=7.5, ha="center")
    a0.set_title(f"{config} — two reactances, no resistors", fontsize=8.5)

    a1 = cplane(fig.add_subplot(gs[1, 0]), max(Rh, Xp) * 1.2)
    phasor(a1, complex(Rh, 0), MUTED, "load")
    Zp = 1 / (1 / (1j * 2 * np.pi * f0 * b) + 1 / Rh) if config.startswith("series") \
        else 1 / (1 / (1j * 2 * np.pi * f0 * b) + 1 / Rh)
    phasor(a1, Zp, ORANGE, "after shunt")
    phasor(a1, Zin0, DOT, "input", lw=2.6)
    a1.set_xlabel("R  (Ω)"); a1.set_ylabel("X  (Ω)")
    a1.set_title("the shunt pulls it down, the series pushes it back",
                 fontsize=8.5)

    a2 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a2.plot(ff / 1e6, VSWR, color=POS, lw=1.7)
    a2.axhline(2.0, color=MUTED, lw=0.8, ls=":")
    a2.axvline(f0_mhz, color=DOT, lw=1.0)
    a2.axvline(f_mhz, color=FG, lw=1.0, ls="--")
    a2.set_ylim(1, 8); a2.set_ylabel("VSWR")
    band = ff[VSWR <= 2.0]
    bw = (band[-1] - band[0]) / 1e6 if len(band) > 1 else 0.0
    a2.set_title(f"VSWR ≤ 2 over {bw:.2f} MHz "
                 f"({100*bw/f0_mhz:.1f}% of f0) — set by Q alone")

    a3 = panel(fig.add_subplot(gs[1, 1]), ORANGE)
    a3.plot(ff / 1e6, Z.real, color=POS, lw=1.5, label="R part")
    a3.plot(ff / 1e6, Z.imag, color=ORANGE, lw=1.5, label="X part")
    a3.axhline(Rl, color=MUTED, lw=0.8, ls=":")
    a3.axhline(0, color=GRIDC, lw=0.8)
    a3.axvline(f0_mhz, color=DOT, lw=1.0)
    a3.set_xlabel("frequency  (MHz)"); a3.set_ylabel("Ω")
    a3.set_ylim(-3 * Rl, 4 * Rl); a3.legend(fontsize=7)
    a3.set_title(f"at f0 exactly {Zin0.real:.4f} {Zin0.imag:+.4f}j Ω")

    readout(fig, 0.845, 0.90, [
        "TRANSFORM", "─" * 26,
        f"from        {Rl:>10.1f}Ω",
        f"to          {Rh:>10.1f}Ω",
        f"ratio       {Rh/Rl:>10.2f}",
        f"f0          {f0_mhz:>10.3f}MHz",
        "", "NETWORK", "─" * 26,
        f"Q           {Q:>10.5f}",
        f"Xs          {Xs:>10.4f}Ω",
        f"Xp          {Xp:>10.4f}Ω",
        (f"L           {a*1e9:>10.3f}nH" if config.startswith("series")
         else f"C           {a*1e12:>10.3f}pF"),
        (f"C           {b*1e12:>10.3f}pF" if config.startswith("series")
         else f"L           {b*1e9:>10.3f}nH"),
        "", "AT f0", "─" * 26,
        f"Zin real    {Zin0.real:>10.6f}Ω",
        f"Zin imag    {Zin0.imag:>+10.6f}Ω",
        f"|Zin−Rl|    {abs(Zin0-Rl):>10.2e}Ω",
        f"|Γ|         {abs((Zin0-Rl)/(Zin0+Rl)):>10.2e}",
        f"VSWR        {1.0:>10.6f}",
        "", "AT MARKER", "─" * 26,
        f"|Γ|         {Gnow:>10.4f}",
        f"VSWR        {(1+Gnow)/max(1-Gnow,1e-12):>10.4f}",
    ], color=GREEN if Gnow < 0.2 else ORANGE)
    footer(fig, f"Q = √(Rh/Rl − 1) = {Q:.4f}   ·   exact at f0, narrower as the "
                f"ratio grows   ·   no resistors, no loss")
    plt.show()


w3 = dict(Rl=widgets.FloatSlider(value=50, min=10, max=200, step=5,
                                 description="source R (Ω):", **SL),
          Rh=widgets.FloatSlider(value=300, min=60, max=3000, step=10,
                                 description="load R (Ω):", **SL),
          f0_mhz=widgets.FloatSlider(value=100, min=1, max=500, step=1,
                                     description="f0 (MHz):", **SL),
          config=widgets.Dropdown(options=["series L, shunt C",
                                           "series C, shunt L"],
                                  value="series L, shunt C",
                                  description="topology:", **SL),
          f_mhz=widgets.FloatSlider(value=100, min=20, max=220, step=1,
                                    description="marker f (MHz):", **SL))
display(widgets.VBox([widgets.HBox([w3["Rl"], w3["Rh"], w3["f0_mhz"]]),
                      widgets.HBox([w3["config"], w3["f_mhz"]])]),
        widgets.interactive_output(draw_match, w3))

Output()

## The transmission line — a ladder long enough that time matters

Take the ladder from two sections ago and make it long. Once the travel time is comparable to a period, "the voltage at the far end" stops being the same quantity as "the voltage at the near end", and the circuit becomes a medium:

$$Z_0=\sqrt{\frac{L}{C}},\qquad
v_p=\frac{1}{\sqrt{LC}},\qquad
\Gamma=\frac{R_L-Z_0}{R_L+Z_0}$$

$Z_0$ is not a resistor. Nothing in the line dissipates, yet a source driving it sees a real resistance, because the energy is leaving and not coming back — at least not yet.

What happens when it reaches the end is the whole subject. The simulation below is an actual LC ladder integrated in time, and the pulse is visibly travelling:

| load | $\Gamma$ | what returns |
|---|---|---|
| $R_L=Z_0$ | $0$ | nothing — the line looks infinite |
| open | $+1$ | the pulse comes back upright, doubling at the end |
| short | $-1$ | it comes back inverted |
| $2Z_0$ | $+1/3$ | a third of it returns |

Press ▶ and watch. A matched line is the only case where the source never learns the line had an end, which is the reason cables are terminated and the reason an unterminated one produces the ringing you see on a mistreated digital edge.

In [ ]:
NSEC = 160
Z0_LINE = 50.0
_L, _C, _DT = Z0_LINE, 1.0 / Z0_LINE, 0.5      # v_p = 1 section per unit time


def line_sim(RL_over_Z0, nsteps=620, drive="pulse", sig=13.0):
    """LC ladder, leapfrog inside, implicit at both terminations."""
    V = np.zeros(NSEC + 1); I = np.zeros(NSEC)
    hist = np.zeros((nsteps, NSEC + 1))
    RL = RL_over_Z0 * Z0_LINE
    a = 2 * _DT / _C
    Rs = Z0_LINE
    for n in range(nsteps):
        src = (np.exp(-((n - 40) / sig) ** 2) if drive == "pulse"
               else np.sin(2 * np.pi * n / 34) * (n < 34 * 6))
        I += (_DT / _L) * (V[:-1] - V[1:])
        V[1:-1] += (_DT / _C) * (I[:-1] - I[1:])
        V[0] = (V[0] + a * (src / Rs - I[0])) / (1 + a / Rs)
        if RL_over_Z0 > 500:
            V[-1] = V[-1] + a * I[-1]                       # open
        elif RL_over_Z0 < 1e-6:
            V[-1] = 0.0                                     # short
        else:
            V[-1] = (V[-1] + a * I[-1]) / (1 + a / RL)      # implicit, any RL
        hist[n] = V
    return hist


LOADS = {"matched  RL = Z0": 1.0, "open circuit": 1e6, "short circuit": 0.0,
         "RL = 2·Z0": 2.0, "RL = Z0/2": 0.5}


def draw_line(k, load, drive):
    RLr = LOADS[load]
    nst = 620
    hist = line_sim(RLr, nst, drive)
    kk = int(min(k, nst - 1))
    V = hist[kk]
    Z0 = Z0_LINE
    probe = hist[:, 50]
    cut = int(np.argmax(np.abs(probe)) + (NSEC - 50) * 1.6)
    inc = np.trapezoid(probe[:cut]); ref = np.trapezoid(probe[cut:])
    G_meas = ref / inc if abs(inc) > 1e-9 else 0.0
    G = (RLr * Z0 - Z0) / (RLr * Z0 + Z0) if RLr < 500 else 1.0
    if RLr < 1e-6:
        G = -1.0
    x = np.arange(NSEC + 1)
    vmax = max(np.abs(hist).max(), 1e-9)

    fig = plt.figure(figsize=(13.0, 5.2))
    gs = fig.add_gridspec(3, 3, width_ratios=[1.35, 1.1, 0.55],
                          height_ratios=[0.8, 1, 1], wspace=0.28, hspace=0.55,
                          left=0.045, right=0.995, top=0.90, bottom=0.10)

    a0 = fig.add_subplot(gs[0, :2])
    panel(a0)
    pts = np.stack([x, np.zeros_like(x)], 1)
    from matplotlib.collections import LineCollection
    segs = np.stack([pts[:-1], pts[1:]], 1)
    cols = [vcolor(v, vmax) for v in V[:-1]]
    a0.add_collection(LineCollection(segs, colors=cols, linewidths=9))
    a0.plot(0, 0, "o", ms=9, color=DOT)
    a0.plot(NSEC, 0, "s", ms=10,
            color=GREEN if abs(G) < 1e-9 else (NEG if G < 0 else ORANGE))
    a0.set_xlim(-3, NSEC + 3); a0.set_ylim(-1, 1)
    a0.set_yticks([]); a0.set_xticks([])
    a0.set_title("the line itself, coloured by voltage — source left, load right")

    a1 = panel(fig.add_subplot(gs[1, :2]), BLUE)
    a1.plot(x, V, color=POS, lw=1.6)
    a1.axhline(0, color=GRIDC, lw=0.8)
    a1.set_ylim(-2.2, 2.2); a1.set_xlim(0, NSEC)
    a1.set_ylabel("v along the line")
    a1.set_title(f"{load} — Γ = {G:+.4f}")

    a2 = panel(fig.add_subplot(gs[2, :2]), ORANGE)
    a2.imshow(hist.T, aspect="auto", origin="lower", cmap=VMAP,
              vmin=-vmax, vmax=vmax,
              extent=[0, nst, 0, NSEC])
    a2.axvline(kk, color=FG, lw=1.0)
    a2.set_xlabel("time step"); a2.set_ylabel("position")
    a2.grid(False)
    a2.set_title("the same thing as a space–time map — slope is the wave speed")

    vin = hist[:kk + 1, 2]
    readout(fig, 0.845, 0.90, [
        "LINE", "─" * 26,
        f"sections    {NSEC:>10d}",
        f"Z0          {Z0:>10.1f}Ω",
        f"drive       {drive:>14s}",
        "", "LOAD", "─" * 26,
        f"{load:>26s}",
        f"RL / Z0     {(RLr if RLr<500 else np.inf):>10.3f}",
        f"Γ           {G:>+10.4f}",
        f"|Γ|         {abs(G):>10.4f}",
        f"VSWR        {((1+abs(G))/(1-abs(G)) if abs(G)<1 else np.inf):>10.3f}",
        f"power sent  {100*(1-G**2):>10.1f}%",
        f"reflected   {100*G**2:>10.1f}%",
        "", "NOW", "─" * 26,
        f"step        {kk:>10d}",
        f"v at input  {V[2]:>+10.4f}",
        f"v at load   {V[-1]:>+10.4f}",
        f"peak on line{np.abs(V).max():>10.4f}",
        "", "MEASURED", "─" * 26,
        f"Γ from area {G_meas:>+10.4f}",
        f"vs theory   {abs(G_meas-G):>10.4f}",
    ], color=GREEN if abs(G) < 1e-9 else (NEG if abs(G) > 0.9 else FG))
    footer(fig, f"LC ladder, {NSEC} sections, implicit terminations   ·   "
                f"Γ = (RL−Z0)/(RL+Z0) = {G:+.3f}   ·   leapfrog integration")
    plt.show()


_p4, _s4 = timeline(619, step=5, interval=55)
w4 = dict(load=widgets.Dropdown(options=list(LOADS), value="open circuit",
                                description="load:", **SL),
          drive=widgets.Dropdown(options=["pulse", "burst"], value="pulse",
                                 description="drive:", **SL),
          k=_s4)
display(widgets.VBox([widgets.HBox([w4["load"], w4["drive"]]),
                      widgets.HBox([_p4, _s4])]),
        widgets.interactive_output(draw_line, w4))

Output()

## The envelope detector — getting the message back off the carrier

An AM signal is a carrier whose amplitude carries the message. No linear circuit can extract it, because the information sits at the *envelope*, not at any frequency present. A diode and an RC do it in two moves: rectify, then smooth.

The capacitor charges to each carrier peak through the diode and discharges through $R$ between peaks, so the output rides the envelope — provided $RC$ lands in the window

$$\frac{1}{f_c}\ \ll\ RC\ \ll\ \frac{1}{f_m}$$

Both failures are visible and both are measured in the panel. Too small and the capacitor follows the individual carrier cycles, leaving ripple: at $RC=0.2$ µs the carrier component is **2.8×** the audio and the tracking error is 98%. Too large and it cannot discharge fast enough to follow the envelope down, giving diagonal clipping: at $RC=200$ µs the ripple is negligible but the error is back up to **39%**. In between, $RC=20$ µs gives 3.9% ripple and 4.9% error.

The other limit is modulation depth. Above $m=1$ the envelope would have to go negative, the carrier phase inverts, and the detector — which only ever sees magnitude — folds the message. Push the slider past 1 and watch the recovered audio distort.

In [6]:
def am_detect(fc_khz, fm_khz, m, RC_us, nper=6):
    fc, fm = fc_khz * 1e3, fm_khz * 1e3
    fs = 60 * fc
    t = np.arange(0, nper / fm, 1 / fs)
    env = 1 + m * np.cos(2 * np.pi * fm * t)
    rf = env * np.cos(2 * np.pi * fc * t)
    RC = RC_us * 1e-6
    out = np.empty_like(rf)
    v = 0.0
    dt = 1 / fs
    for n, xx in enumerate(rf):
        if xx > v:
            v = xx
        else:
            v -= v / RC * dt
        out[n] = v
    return t, rf, env, out, fs


def draw_detector(fc_khz, fm_khz, m, RC_us):
    t, rf, env, out, fs = am_detect(fc_khz, fm_khz, m, RC_us)
    n0 = len(t) // 3
    o, e = out[n0:], env[n0:]
    F = np.abs(np.fft.rfft(o - o.mean()))
    fr = np.fft.rfftfreq(len(o), 1 / fs)
    rip = F[np.argmin(abs(fr - fc_khz * 1e3))] / max(
        F[np.argmin(abs(fr - fm_khz * 1e3))], 1e-12)
    err = np.max(np.abs(o - e)) / np.max(e)
    vmax = 1 + m

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.1, 1.3, 0.55],
                          wspace=0.28, hspace=0.46, left=0.03, right=0.995,
                          top=0.90, bottom=0.11)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.5, 4.6), (-0.5, 2.4))
    wire(a0, [(0.4, 1.9), (1.2, 1.9)], 1.0, vmax)
    source(a0, (0.4, 0.2), (0.4, 1.9), 0.6, vmax, "ac", "AM in")
    diode(a0, (1.2, 1.9), (2.6, 1.9), 1.0, vmax, "D")
    wire(a0, [(2.6, 1.9), (4.2, 1.9)], out[-1] / vmax, vmax)
    resistor(a0, (3.2, 1.9), (3.2, 0.2), 0.4, vmax, "R")
    capacitor(a0, (4.2, 1.9), (4.2, 0.2), 0.4, vmax, "C")
    wire(a0, [(0.4, 0.2), (4.2, 0.2)], 0.0, vmax)
    node_dot(a0, (3.2, 1.9), out[-1] / vmax, vmax)
    a0.text(3.2, 2.15, "audio out", color=FG, fontsize=8, ha="center")
    a0.set_title(f"rectify, then smooth — RC = {RC_us:.2f} µs", fontsize=8.5)

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    sl = slice(0, len(t) // 3)
    a1.plot(t[sl] * 1e6, rf[sl], color=MUTED, lw=0.4)
    a1.plot(t[sl] * 1e6, env[sl], color=POS, lw=1.4, label="envelope")
    a1.plot(t[sl] * 1e6, -env[sl], color=POS, lw=0.8, alpha=0.4)
    a1.plot(t[sl] * 1e6, out[sl], color=DOT, lw=1.4, label="detector out")
    a1.set_ylabel("volts"); a1.legend(fontsize=7)
    a1.set_title(f"AM in, m = {m:.2f}")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    a2.plot(t[n0:] * 1e6, e, color=POS, lw=1.6, label="true envelope")
    a2.plot(t[n0:] * 1e6, o, color=DOT, lw=1.3, label="recovered")
    a2.set_xlabel("time  (µs)"); a2.set_ylabel("volts"); a2.legend(fontsize=7)
    verdict = ("carrier ripple" if rip > 0.05 else
               "lags the envelope" if err > 0.15 else "good")
    a2.set_title(f"ripple {rip:.4f} · tracking error {err*100:.2f}% → {verdict}")

    readout(fig, 0.845, 0.90, [
        "SIGNAL", "─" * 26,
        f"carrier     {fc_khz:>10.1f}kHz",
        f"message     {fm_khz:>10.2f}kHz",
        f"depth m     {m:>10.2f}",
        f"ratio fc/fm {fc_khz/fm_khz:>10.1f}",
        "", "DETECTOR", "─" * 26,
        f"RC          {RC_us:>10.2f}µs",
        f"1/fc        {1e3/fc_khz:>10.3f}µs",
        f"1/fm        {1e3/fm_khz:>10.1f}µs",
        "", "WINDOW", "─" * 26,
        f"must exceed {1e3/fc_khz:>10.3f}µs",
        f"must be under{1e3/fm_khz:>9.1f}µs",
        "", "MEASURED", "─" * 26,
        f"ripple      {rip:>10.4f}",
        f"track error {err*100:>10.2f}%",
        f"verdict     {verdict:>14s}",
        "", "m > 1 inverts the",
        "carrier and the",
        "detector folds it",
    ], color=GREEN if verdict == "good" else ORANGE)
    footer(fig, f"1/fc << RC << 1/fm   ·   {1e3/fc_khz:.2f} µs << {RC_us:.2f} µs "
                f"<< {1e3/fm_khz:.1f} µs   ·   peak-following simulation")
    plt.show()


w5 = dict(fc_khz=widgets.FloatSlider(value=1000, min=200, max=2000, step=100,
                                     description="carrier kHz:", **SL),
          fm_khz=widgets.FloatSlider(value=5, min=1, max=20, step=1,
                                     description="message kHz:", **SL),
          m=widgets.FloatSlider(value=0.6, min=0.1, max=1.4, step=0.05,
                                description="depth m:", **SL),
          RC_us=widgets.FloatSlider(value=20, min=0.2, max=200, step=0.2,
                                    description="RC (µs):", **SL))
display(widgets.HBox([w5["fc_khz"], w5["fm_khz"], w5["m"], w5["RC_us"]]),
        widgets.interactive_output(draw_detector, w5))

Output()

## The mixer — moving a signal to a different frequency

Multiplying two sinusoids produces neither of them:

$$\cos(2\pi f_1t)\cos(2\pi f_2t)=\tfrac12\cos\big(2\pi(f_1-f_2)t\big)+\tfrac12\cos\big(2\pi(f_1+f_2)t\big)$$

Verified in the spectrum panel: 10 MHz against 9 MHz gives peaks at exactly **1.000 and 19.000 MHz** and nothing at 9 or 10. No linear circuit can do this — a filter can only remove what is already there — so the mixer is the second essential nonlinear block after the detector.

This is what makes the **superheterodyne** receiver work, and why nearly every radio built since 1918 is one. Rather than building a sharp tunable filter at whatever frequency you want to receive, you move the wanted signal down to a fixed intermediate frequency and put one excellent fixed filter there. Tuning becomes moving the local oscillator, and the hard filtering is done once.

The catch is on the panel as a second marker. Two input frequencies map to the same IF: $f_{LO}+f_{IF}$ and $f_{LO}-f_{IF}$. The one you want and the **image** you do not. Nothing after the mixer can separate them because they arrive on top of each other, so the image has to be removed *before* — which is what the RF filter at the front of every receiver is for, and why the choice of IF is a compromise between image rejection and filter cost.

In [7]:
def draw_mixer(f_rf, f_lo, f_img_on, show_span):
    fs = 200e6
    t = np.arange(0, 3e-4, 1 / fs)
    rf = np.cos(2 * np.pi * f_rf * 1e6 * t)
    f_if = abs(f_rf - f_lo)
    f_image = 2 * f_lo - f_rf
    if f_img_on:
        rf = rf + 0.7 * np.cos(2 * np.pi * f_image * 1e6 * t + 1.1)
    lo = np.cos(2 * np.pi * f_lo * 1e6 * t)
    mx = rf * lo
    win = np.hanning(len(t))
    def spec(x):
        X = np.abs(np.fft.rfft(x * win))
        return np.fft.rfftfreq(len(t), 1 / fs) / 1e6, X / max(X.max(), 1e-12)
    fr, Srf = spec(rf)
    _, Smx = spec(mx)

    fig = plt.figure(figsize=(13.0, 5.2))
    gs = fig.add_gridspec(3, 3, width_ratios=[1.0, 1.35, 0.55],
                          wspace=0.28, hspace=0.55, left=0.03, right=0.995,
                          top=0.90, bottom=0.10)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.5, 4.2), (-0.6, 3.0))
    a0.add_patch(mpatches.Circle((2.0, 1.8), 0.45, fill=False, ec=FG, lw=2.0))
    a0.plot([1.72, 2.28], [1.52, 2.08], color=FG, lw=1.8)
    a0.plot([1.72, 2.28], [2.08, 1.52], color=FG, lw=1.8)
    wire(a0, [(0.2, 1.8), (1.55, 1.8)], 0.8, 1.0)
    a0.text(0.2, 2.05, f"RF {f_rf:.1f} MHz", color=POS, fontsize=7.5)
    wire(a0, [(2.0, 0.5), (2.0, 1.35)], 0.4, 1.0)
    a0.text(2.0, 0.25, f"LO {f_lo:.1f} MHz", color=GREEN, fontsize=7.5,
            ha="center")
    wire(a0, [(2.45, 1.8), (3.9, 1.8)], 0.3, 1.0)
    a0.text(3.9, 2.05, f"IF {f_if:.1f} MHz", color=DOT, fontsize=7.5, ha="right")
    if f_img_on:
        a0.text(0.2, 2.45, f"image {f_image:.1f} MHz", color=NEG, fontsize=7.5)
    a0.set_title("a multiplier — nothing else needed", fontsize=8.5)

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    a1.plot(fr, 20 * np.log10(np.maximum(Srf, 1e-5)), color=POS, lw=1.0)
    a1.axvline(f_lo, color=GREEN, lw=1.1, ls="--")
    a1.text(f_lo, 3, " LO", color=GREEN, fontsize=7)
    if f_img_on:
        a1.axvline(f_image, color=NEG, lw=1.0, ls=":")
    a1.set_xlim(0, show_span); a1.set_ylim(-60, 8)
    a1.set_ylabel("dB"); a1.set_title("what arrives at the mixer")

    a2 = panel(fig.add_subplot(gs[1, 1]), ORANGE)
    a2.plot(fr, 20 * np.log10(np.maximum(Smx, 1e-5)), color=ORANGE, lw=1.0)
    a2.axvline(f_if, color=DOT, lw=1.1, ls="--")
    a2.text(f_if, 3, " IF", color=DOT, fontsize=7)
    a2.axvline(f_rf + f_lo, color=MUTED, lw=1.0, ls=":")
    a2.text(f_rf + f_lo, 3, " sum", color=MUTED, fontsize=7)
    a2.set_xlim(0, show_span); a2.set_ylim(-60, 8)
    a2.set_ylabel("dB"); a2.set_title("difference and sum — the inputs are gone")

    a3 = panel(fig.add_subplot(gs[2, 1]), GREEN)
    sl = slice(0, 1200)
    a3.plot(t[sl] * 1e6, rf[sl], color=POS, lw=0.7, label="RF")
    a3.plot(t[sl] * 1e6, mx[sl], color=ORANGE, lw=0.9, label="product")
    a3.set_xlabel("time  (µs)"); a3.legend(fontsize=7)
    a3.set_title("the product in time")

    pk = [fr[i] for i in range(1, len(Smx) - 1)
          if Smx[i] > Smx[i - 1] and Smx[i] >= Smx[i + 1] and Smx[i] > 0.15]
    readout(fig, 0.845, 0.90, [
        "FREQUENCIES", "─" * 26,
        f"RF          {f_rf:>10.3f}MHz",
        f"LO          {f_lo:>10.3f}MHz",
        f"IF = |Δ|    {f_if:>10.3f}MHz",
        f"sum         {f_rf+f_lo:>10.3f}MHz",
        "", "MEASURED PEAKS", "─" * 26,
    ] + [f"            {p:>10.3f}MHz" for p in pk[:4]] + [
        "", "IMAGE", "─" * 26,
        f"image freq  {f_image:>10.3f}MHz",
        f"present     {str(bool(f_img_on)):>14s}",
        f"separation  {abs(f_rf-f_image):>10.3f}MHz",
        "", "the image lands on the",
        "same IF — only an RF",
        "filter BEFORE the mixer",
        "can remove it",
    ], color=NEG if f_img_on else FG)
    footer(fig, "cos a · cos b = ½[cos(a−b) + cos(a+b)]   ·   "
                "superheterodyne: tune the LO, filter once at the IF")
    plt.show()


w6 = dict(f_rf=widgets.FloatSlider(value=10, min=1, max=40, step=0.5,
                                   description="RF (MHz):", **SL),
          f_lo=widgets.FloatSlider(value=9, min=1, max=40, step=0.5,
                                   description="LO (MHz):", **SL),
          f_img_on=widgets.Checkbox(value=False, description="add the image signal",
                                    indent=False),
          show_span=widgets.FloatSlider(value=60, min=20, max=100, step=5,
                                        description="span (MHz):", **SL))
display(widgets.HBox([w6["f_rf"], w6["f_lo"], w6["show_span"], w6["f_img_on"]]),
        widgets.interactive_output(draw_mixer, w6))

Output()

## From the feed to the antenna — where the circuit stops being a circuit

Every circuit so far has been small compared with a wavelength, so the same current flowed everywhere in a branch. An antenna is deliberately the opposite: it is made a *significant fraction of a wavelength* long, and the current then varies along it.

$$I(z)=I_0\sin\!\left(k\!\left(\frac{L}{2}-|z|\right)\right),\qquad k=\frac{2\pi}{\lambda}$$

For a half-wave dipole that is a half sine — maximum at the feed, zero at the tips, which it must be since charge has nowhere further to go. Make it a full wavelength and the current at the feed becomes **zero**, so the feedpoint impedance goes enormous; that is why antenna length is not a free parameter.

To the transmitter this looks like an ordinary impedance, $Z_a=R_a+jX_a$, and for a half-wave dipole in free space $R_a\approx73\ \Omega$ with $X_a\approx+42\ \Omega$. Trimming slightly shorter than $\lambda/2$ cancels the reactance, which is why real dipoles are cut to about $0.475\lambda$.

$R_a$ is the strange part. It is a genuine resistance — it dissipates power out of the circuit exactly as a resistor would, and the readout shows the feed current following $P=\frac12I^2R_a$. But nothing gets hot. The power leaves as a wave, and the next section is where it goes.

In [ ]:
FIELDMAP = mpl.colors.LinearSegmentedColormap.from_list("emfield", [
    (0.00, "#eaf3ff"), (0.16, "#5aa9e6"), (0.44, "#0a1622"), (0.50, "#05070b"),
    (0.56, "#211307"), (0.84, "#e08a3c"), (1.00, "#fff3e2")])
ETA0 = 376.730313
C_LIGHT = 2.998e8


def dipole_current(z, Lr):
    """Standing-wave current along a centre-fed dipole, z and L in wavelengths."""
    return np.sin(2 * np.pi * (Lr / 2 - np.abs(z)))


def dipole_Z(Lr):
    """Feedpoint impedance, anchored on the standard half-wave values."""
    Ifeed = np.sin(2 * np.pi * Lr / 2)
    Ra = 73.1 * (Ifeed ** 2) / max(np.sin(np.pi / 2) ** 2, 1e-9)
    Ra = max(Ra, 1e-3) if abs(Ifeed) > 1e-3 else 1e-3
    Xa = 42.5 + 900 * (Lr - 0.5)
    return Ra, Xa


def draw_feed(k, Lr, Vs, f_mhz, match_on):
    lam = C_LIGHT / (f_mhz * 1e6)
    Ra, Xa = dipole_Z(Lr)
    Zs = 50.0
    if match_on:
        Zin = complex(Zs, 0.0)
    else:
        Zin = complex(Ra, Xa)
    I = Vs / (Zs + Zin)
    Pdel = 0.5 * abs(I) ** 2 * Zin.real
    G = abs((Zin - Zs) / (Zin + Zs))
    wt = 2 * np.pi * k / 120
    z = np.linspace(-Lr / 2, Lr / 2, 240)
    Iz = dipole_current(z, Lr) * np.cos(wt)
    vmax = max(Vs, 1e-9)

    fig = plt.figure(figsize=(13.0, 5.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.15, 0.55],
                          wspace=0.28, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.10)

    a0 = fig.add_subplot(gs[:, 0])
    sch_axes(a0, (-0.6, 4.8), (-2.6, 2.6))
    wire(a0, [(0.3, 0.35), (2.1, 0.35)], Vs / 2, vmax)
    wire(a0, [(0.3, -0.35), (2.1, -0.35)], -Vs / 2, vmax)
    source(a0, (0.3, -0.35), (0.3, 0.35), 0.0, vmax, "ac", f"{Vs:.1f} V")
    if match_on:
        inductor(a0, (1.0, 0.35), (1.8, 0.35), Vs / 3, vmax, "match")
    resistor(a0, (2.1, 0.35), (2.1, -0.35), 0.0, vmax, None)
    a0.text(2.45, 0.0, f"Ra {Ra:.1f}Ω\nXa {Xa:+.1f}Ω", color=FG, fontsize=7,
            va="center")
    zt = np.linspace(0.06, Lr / 2 * 4, 60)
    for sgn in (1, -1):
        a0.plot([3.6, 3.6], [sgn * 0.08, sgn * Lr / 2 * 4], color=FG, lw=3.0)
    prof = dipole_current(np.linspace(-Lr / 2, Lr / 2, 120), Lr)
    yy = np.linspace(-Lr / 2, Lr / 2, 120) * 4
    a0.fill_betweenx(yy, 3.6, 3.6 + prof * np.cos(wt) * 0.7, color=DOT, alpha=0.45)
    a0.plot(3.6 + prof * np.cos(wt) * 0.7, yy, color=DOT, lw=1.4)
    wire(a0, [(2.1, 0.35), (3.6, 0.08)], Vs / 2, vmax)
    wire(a0, [(2.1, -0.35), (3.6, -0.08)], -Vs / 2, vmax)
    charge_dots(a0, seg((0.3, 0.35), (2.1, 0.35), 30), k / 120 * abs(I) * 60,
                spacing=0.26, ms=3.6)
    charge_dots(a0, seg((2.1, -0.35), (0.3, -0.35), 30), k / 120 * abs(I) * 60,
                spacing=0.26, ms=3.6)
    a0.text(3.6, Lr / 2 * 4 + 0.25, f"{Lr:.3f}λ dipole", color=FG, fontsize=8,
            ha="center")
    a0.set_title("the transmitter, and the current standing wave on the wire")

    a1 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    for ph, al in ((0, 1.0), (np.pi / 3, 0.35), (2 * np.pi / 3, 0.2)):
        a1.plot(z, dipole_current(z, Lr) * np.cos(wt + ph), color=DOT,
                lw=1.6 if ph == 0 else 0.9, alpha=al)
    a1.plot(z, dipole_current(z, Lr), color=POS, lw=0.9, ls="--",
            label="envelope")
    a1.plot(z, -dipole_current(z, Lr), color=POS, lw=0.9, ls="--")
    a1.axvline(0, color=GRIDC, lw=0.8); a1.axhline(0, color=GRIDC, lw=0.8)
    a1.set_xlabel("position along the dipole  (λ)"); a1.set_ylabel("current")
    a1.legend(fontsize=7)
    a1.set_title(f"feed current {abs(dipole_current(np.array([0.0]), Lr))[0]:.4f} "
                 f"·  tips always zero")

    a2 = panel(fig.add_subplot(gs[1, 1]), ORANGE)
    LL = np.linspace(0.05, 1.05, 400)
    Ras = np.array([dipole_Z(l)[0] for l in LL])
    Xas = np.array([dipole_Z(l)[1] for l in LL])
    a2.plot(LL, Ras, color=POS, lw=1.6, label="Ra")
    a2.plot(LL, Xas, color=ORANGE, lw=1.4, label="Xa")
    a2.axhline(0, color=GRIDC, lw=0.8)
    a2.axvline(Lr, color=FG, lw=1.0, ls="--")
    a2.axvline(0.5, color=DOT, lw=0.9, ls=":")
    a2.set_ylim(-200, 250); a2.set_xlabel("dipole length  (λ)")
    a2.set_ylabel("Ω"); a2.legend(fontsize=7)
    a2.set_title("feedpoint impedance vs length — 73 Ω at a half wave")

    readout(fig, 0.845, 0.90, [
        "TRANSMITTER", "─" * 26,
        f"frequency   {f_mhz:>10.1f}MHz",
        f"λ           {lam:>10.3f}m",
        f"source      {Vs:>10.2f}V",
        f"source Z    {Zs:>10.1f}Ω",
        f"matching    {str(bool(match_on)):>14s}",
        "", "ANTENNA", "─" * 26,
        f"length      {Lr:>10.3f}λ",
        f"physical    {Lr*lam:>10.3f}m",
        f"Ra          {Ra:>10.2f}Ω",
        f"Xa          {Xa:>+10.2f}Ω",
        f"|Γ| to 50Ω  {G:>10.4f}",
        f"VSWR        {(1+G)/max(1-G,1e-9):>10.3f}",
        "", "POWER", "─" * 26,
        f"feed current{abs(I)*1e3:>10.3f}mA",
        f"P = ½I²Ra   {Pdel*1e3:>10.4f}mW",
        f"radiated    {Pdel*1e3:>10.4f}mW",
        f"heat in Ra  {0.0:>10.4f}mW",
        "", "Ra dissipates power",
        "but nothing warms up",
    ], color=GREEN if G < 0.2 else ORANGE)
    footer(fig, f"I(z) = I0 sin(k(L/2−|z|))   ·   Ra ≈ 73 Ω at λ/2   ·   "
                f"P = ½I²Ra leaves as a wave")
    plt.show()


_pA, _sA = timeline(119, step=2)
wA = dict(Lr=widgets.FloatSlider(value=0.5, min=0.05, max=1.0, step=0.005,
                                 description="length (λ):", **SL),
          Vs=widgets.FloatSlider(value=10, min=1, max=30, step=1,
                                 description="source V:", **SL),
          f_mhz=widgets.FloatSlider(value=100, min=10, max=1000, step=10,
                                    description="frequency MHz:", **SL),
          match_on=widgets.Checkbox(value=False, description="matched to 50 Ω",
                                    indent=False),
          k=_sA)
display(widgets.VBox([widgets.HBox([wA["Lr"], wA["Vs"], wA["f_mhz"]]),
                      widgets.HBox([wA["match_on"], _pA, _sA])]),
        widgets.interactive_output(draw_feed, wA))

Output()

## The dipole radiating — where the power actually goes

The current on the wire is now the *source term* for a field. Because the field is axially symmetric, the electric field lines are contours of a stream function, with $u=kr$:

$$\Psi(u,\theta,t)=\sin^2\!\theta\left[\underbrace{\frac{\cos(\omega t-u)}{u}}_{\text{bound}}-\underbrace{\vphantom{\frac{1}{u}}\sin(\omega t-u)}_{\text{radiated}}\right]$$

The two terms are equal in size at $u=1$, that is at $r=\lambda/2\pi=0.159\lambda$, and they behave completely differently. Step the phase and watch: loops grow out of the wire, pinch off near that radius, and travel away for ever. Inside it the field is mostly *bound* — it swells and collapses back into the antenna twice per cycle and carries nothing away.

That pinch-off is the radiation, and it is what the 73 Ω is charging for. Select the bound term alone and the loops never detach: energy goes out and comes straight back, which is a reactance, not a resistance.

The right panel is the resulting pattern. A short dipole and a half-wave dipole differ far less than people expect — 1.5 against 1.64 in directivity — because both are essentially $\sin\theta$. What changes hugely with length is the *feedpoint impedance*, not the shape of the radiation.

In [9]:
def draw_radiate(k, rmax, part, nlev, Lr):
    wt = 2 * np.pi * k / 120
    n = 200
    xg = np.linspace(1e-3, rmax, n)
    zg = np.linspace(-rmax, rmax, n)
    X, Z = np.meshgrid(xg, zg)
    R = np.hypot(X, Z)
    U = 2 * np.pi * R
    s2 = (X / R) ** 2
    bound = s2 * np.cos(wt - U) / U
    rad = -s2 * np.sin(wt - U)
    psi = {"full field": bound + rad, "radiated term only": rad,
           "bound term only": bound}[part]
    psi = np.where(R < 0.055, np.nan, psi)
    pos = np.linspace(0.07, 0.85, nlev)
    tmp = plt.figure()
    cs = tmp.add_subplot(111).contour(X, Z, psi,
                                      levels=np.concatenate([-pos[::-1], pos]))
    segs = [(lv, sg) for lv, sl in zip(cs.levels, cs.allsegs) for sg in sl]
    plt.close(tmp)

    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.35, 1.0, 0.55], wspace=0.26,
                          left=0.035, right=0.995, top=0.88, bottom=0.12)

    a0 = panel(fig.add_subplot(gs[0]), BLUE)
    for lv, sg in segs:
        a0.plot(sg[:, 0], sg[:, 1], color=POS if lv > 0 else NEG, lw=0.8,
                alpha=0.85)
        a0.plot(-sg[:, 0], sg[:, 1], color=POS if lv > 0 else NEG, lw=0.8,
                alpha=0.85)
    zz = np.linspace(-Lr / 2, Lr / 2, 80)
    a0.plot(np.zeros_like(zz), zz, color=FG, lw=4)
    a0.plot(dipole_current(zz, Lr) * np.cos(wt) * 0.12, zz, color=DOT, lw=1.6)
    th = np.linspace(0, 2 * np.pi, 120)
    rb = 1 / (2 * np.pi)
    a0.plot(rb * np.cos(th), rb * np.sin(th), color=PURP, lw=1.3, ls="--")
    a0.text(rb * 0.75, rb * 1.25, "r = λ/2π", color=PURP, fontsize=7.5)
    a0.set_xlim(-rmax, rmax); a0.set_ylim(-rmax, rmax)
    a0.set_aspect("equal"); a0.grid(False)
    a0.set_xlabel("x  (λ)"); a0.set_ylabel("z  (λ)")
    a0.set_title(f"electric field lines — ωt = {np.degrees(wt) % 360:.0f}°  ({part})")

    a1 = panel(fig.add_subplot(gs[1], projection="polar"), ORANGE)
    a1.set_facecolor(PANEL)
    th2 = np.linspace(0, 2 * np.pi, 721)
    st = np.abs(np.sin(th2))
    kl = np.pi * Lr
    with np.errstate(divide="ignore", invalid="ignore"):
        F = np.abs((np.cos(kl * np.cos(th2)) - np.cos(kl)) / np.where(st < 1e-6, 1, st))
    F = np.where(st < 1e-6, 0, F)
    F = F / max(F.max(), 1e-9)
    a1.plot(th2, F, color=ORANGE, lw=1.6)
    a1.fill_between(th2, 0, F, color=ORANGE, alpha=0.2)
    a1.plot(th2, st / st.max(), color=MUTED, lw=0.9, ls="--")
    a1.set_theta_zero_location("N")
    a1.set_ylim(0, 1.05); a1.tick_params(colors=MUTED, labelsize=6.5)
    a1.grid(alpha=0.2, color=GRIDC)
    a1.set_title("pattern — dashed is a short dipole (sinθ)", pad=14)

    Dhw = 1.64
    readout(fig, 0.845, 0.88, [
        "FIELD REGIONS", "─" * 26,
        f"reactive    r < {1/(2*np.pi):.4f}λ",
        f"crossover   u = kr = 1",
        f"shown out to{rmax:>10.2f}λ",
        f"phase       {np.degrees(wt)%360:>10.0f}°",
        "", "TERMS", "─" * 26,
        "bound     ~ 1/u",
        "radiated  ~ 1",
        "equal at r = λ/2π",
        "", "PATTERN", "─" * 26,
        f"length      {Lr:>10.3f}λ",
        f"directivity {(1.5 if Lr<0.2 else Dhw):>10.2f}",
        f"in dBi      {10*np.log10(1.5 if Lr<0.2 else Dhw):>10.2f}",
        "", "short dipole 1.50",
        "half wave    1.64",
        "the shape barely",
        "changes — the",
        "impedance changes",
        "enormously",
    ])
    footer(fig, "Ψ = sin²θ[cos(ωt−u)/u − sin(ωt−u)]   ·   "
                "loops pinch off at r ≈ λ/2π and never come back")
    plt.show()


_pB, _sB = timeline(119, step=2)
wB = dict(rmax=widgets.FloatSlider(value=2.0, min=0.8, max=4.0, step=0.2,
                                   description="extent (λ):", **SL),
          part=widgets.Dropdown(options=["full field", "radiated term only",
                                         "bound term only"],
                                value="full field", description="show:", **SL),
          nlev=widgets.IntSlider(value=5, min=3, max=9, step=1,
                                 description="line density:", **SL),
          Lr=widgets.FloatSlider(value=0.5, min=0.05, max=1.0, step=0.05,
                                 description="length (λ):", **SL),
          k=_sB)
display(widgets.VBox([widgets.HBox([wB["rmax"], wB["part"], wB["nlev"]]),
                      widgets.HBox([wB["Lr"], _pB, _sB])]),
        widgets.interactive_output(draw_radiate, wB))

Output()

## The crossing — one wave, two antennas, and a delay

Put a second dipole some distance away and the field takes time to reach it. The picture below is the instantaneous $E_z$ of the wave leaving the transmitter, and the receiving element only starts producing anything after

$$t_d=\frac{R}{c}$$

which for the default 10 m at 100 MHz is 33.4 ns, a third of a period. The trace under the field shows the transmit current and the induced voltage on the same axis, and the gap between them opening as you move the receiver away is the propagation delay — the same thing radar measures for range.

Two things are worth watching. Move the receiver out and the amplitude falls as $1/R$ — the *field* falls as $1/R$, and power therefore as $1/R^2$, which is the one-way spreading loss from the EW notebook arriving here from the circuit side.

Then rotate the receiving dipole. At 90° to the transmitter it picks up nothing at all, because the induced voltage depends on the component of $\mathbf{E}$ along the wire:

$$V_{oc}=\mathbf{E}\cdot\mathbf{h}_{eff}=E\,h_{eff}\cos\psi$$

Cross-polarisation is not attenuation, it is a dot product going to zero — which is why polarisation is a resource that can be reused, and why a misaligned antenna can be deaf to a very strong signal.

In [ ]:
def draw_link(k, R_m, f_mhz, tilt_deg, Pt_W):
    lam = C_LIGHT / (f_mhz * 1e6)
    Rl = R_m / lam
    wt = 2 * np.pi * k / 140
    G = 1.64
    Ra = 73.1
    heff = lam / np.pi
    E0 = np.sqrt(2 * ETA0 * Pt_W * G / (4 * np.pi * R_m ** 2))
    Voc = E0 * heff * np.cos(np.deg2rad(tilt_deg))
    td = R_m / C_LIGHT

    ext = max(Rl * 1.35, 2.0)
    n = 220
    xs = np.linspace(-0.35 * ext, ext, n)
    zs = np.linspace(-ext * 0.62, ext * 0.62, n)
    X, Zc = np.meshgrid(xs, zs)
    Rg = np.hypot(X, Zc)
    st2 = np.where(Rg > 1e-6, (X / np.maximum(Rg, 1e-9)) ** 2, 0.0)
    Ez = np.where(Rg > 0.08,
                  np.sqrt(st2) * np.cos(wt - 2 * np.pi * Rg) / np.maximum(Rg, 0.08),
                  0.0)
    sc = np.percentile(np.abs(Ez), 99) or 1.0

    fig = plt.figure(figsize=(13.0, 5.4))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.5, 1.0, 0.55],
                          height_ratios=[1.25, 1], wspace=0.26, hspace=0.42,
                          left=0.035, right=0.995, top=0.90, bottom=0.10)

    a0 = panel(fig.add_subplot(gs[:, 0]), BLUE)
    a0.imshow(Ez, extent=[xs[0], xs[-1], zs[0], zs[-1]], origin="lower",
              cmap=FIELDMAP, vmin=-sc, vmax=sc, aspect="equal",
              interpolation="bilinear")
    zz = np.linspace(-0.25, 0.25, 40)
    a0.plot(np.zeros_like(zz), zz, color="#f2f5fa", lw=4)
    a0.plot(dipole_current(zz, 0.5) * np.cos(wt) * 0.18, zz, color=DOT, lw=1.6)
    tl = np.deg2rad(tilt_deg)
    a0.plot([Rl - 0.25 * np.sin(tl), Rl + 0.25 * np.sin(tl)],
            [-0.25 * np.cos(tl), 0.25 * np.cos(tl)], color=GREEN, lw=4)
    a0.plot([0, Rl], [0, 0], color=CYAN if False else PURP, lw=0.8, ls=":")
    a0.text(0, -0.42, "TX", color="#f2f5fa", fontsize=8, ha="center")
    a0.text(Rl, -0.42, "RX", color=GREEN, fontsize=8, ha="center")
    a0.grid(False)
    a0.set_xlabel("distance  (λ)"); a0.set_ylabel("(λ)")
    a0.set_title(f"instantaneous field — {R_m:.1f} m = {Rl:.2f}λ apart, "
                 f"delay {td*1e9:.1f} ns")

    a1 = panel(fig.add_subplot(gs[0, 1]), ORANGE)
    t = np.linspace(0, 3 / (f_mhz * 1e6), 700)
    itx = np.cos(2 * np.pi * f_mhz * 1e6 * t)
    vrx = np.where(t > td, np.cos(2 * np.pi * f_mhz * 1e6 * (t - td)), 0.0)
    a1.plot(t * 1e9, itx, color=DOT, lw=1.4, label="TX current")
    a1.plot(t * 1e9, vrx, color=GREEN, lw=1.4, label="RX voltage")
    a1.axvline(td * 1e9, color=PURP, lw=1.0, ls="--")
    a1.text(td * 1e9, 1.12, f" {td*1e9:.1f} ns", color=PURP, fontsize=7)
    a1.axvline(k / 140 * 3 / (f_mhz * 1e6) * 1e9 % (t[-1] * 1e9), color=FG,
               lw=0.9, ls=":")
    a1.set_ylim(-1.3, 1.35); a1.set_xlabel("time  (ns)")
    a1.legend(fontsize=7); a1.set_title("the delay is the distance")

    a2 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    tt = np.linspace(0, 90, 200)
    a2.plot(tt, np.cos(np.deg2rad(tt)) ** 2, color=GREEN, lw=1.6)
    a2.axvline(tilt_deg, color=FG, lw=1.0, ls="--")
    a2.plot([tilt_deg], [np.cos(tl) ** 2], "o", ms=7, color=DOT)
    a2.set_xlabel("polarisation mismatch  (deg)")
    a2.set_ylabel("power fraction")
    a2.set_title(f"cos²ψ — {100*np.cos(tl)**2:.1f}% of the power gets in")

    readout(fig, 0.845, 0.90, [
        "LINK", "─" * 26,
        f"frequency   {f_mhz:>10.1f}MHz",
        f"λ           {lam:>10.3f}m",
        f"distance    {R_m:>10.2f}m",
        f"in wavel.   {Rl:>10.3f}λ",
        f"delay R/c   {td*1e9:>10.2f}ns",
        f"in cycles   {td*f_mhz*1e6:>10.3f}",
        "", "FIELD AT RX", "─" * 26,
        f"Pt          {Pt_W:>10.2f}W",
        f"|E|         {E0*1e3:>10.4f}mV/m",
        f"S = E²/2η   {E0**2/(2*ETA0)*1e6:>10.4f}µW/m²",
        "", "INDUCED", "─" * 26,
        f"h_eff = λ/π {heff:>10.4f}m",
        f"tilt ψ      {tilt_deg:>10.1f}°",
        f"cos ψ       {np.cos(tl):>10.4f}",
        f"Voc         {Voc*1e3:>10.4f}mV",
        "", f"E falls as 1/R",
        f"power as 1/R²",
        "cross-pol is a dot",
        "product, not a loss",
    ], color=NEG if abs(tilt_deg) > 75 else FG)
    footer(fig, f"Voc = E·h_eff·cosψ   ·   h_eff = λ/π for a half wave   ·   "
                f"delay = R/c = {td*1e9:.2f} ns")
    plt.show()


_pC, _sC = timeline(139, step=2)
wC = dict(R_m=widgets.FloatSlider(value=10, min=2, max=40, step=0.5,
                                  description="distance (m):", **SL),
          f_mhz=widgets.FloatSlider(value=100, min=50, max=300, step=10,
                                    description="frequency MHz:", **SL),
          tilt_deg=widgets.FloatSlider(value=0, min=0, max=90, step=5,
                                       description="RX tilt (°):", **SL),
          Pt_W=widgets.FloatSlider(value=1.0, min=0.1, max=10, step=0.1,
                                   description="Pt (W):", **SL),
          k=_sC)
display(widgets.VBox([widgets.HBox([wC["R_m"], wC["f_mhz"], wC["tilt_deg"]]),
                      widgets.HBox([wC["Pt_W"], _pC, _sC])]),
        widgets.interactive_output(draw_link, wC))

Output()

## The receiving antenna — a Thévenin source you cannot see

The wave arrives, drives current along the receiving wire, and from the receiver's point of view the antenna is nothing more exotic than a source with an internal impedance — Thévenin, exactly as in the theory notebook:

$$V_{oc}=E\,h_{eff},\qquad Z_a=R_a+jX_a$$

So the whole of maximum power transfer applies unchanged. The load should be the **conjugate** $R_a-jX_a$, and then

$$P_{load}=\frac{|V_{oc}|^2}{8R_a}$$

Here is the check that ties the two halves of this series together. That expression is pure circuit theory — a voltage source, an internal resistance, a matched load. The field-side answer is $P=S\cdot A_e$ with $S=E^2/2\eta$ and $A_e=G\lambda^2/4\pi$, and it contains no circuit quantities at all. Computed both ways in the readout they agree to **0.03%**, the residual being the rounding in $G=1.64$ and $R_a=73.1$.

Half the available power is unavoidably lost even at a perfect match, exactly as in the theory notebook — the antenna re-radiates it. And the mismatch curve is the same shape as before: drag the load away from $R_a$ in either direction and the delivered power falls symmetrically.

The rest is the receiver you already built: match, filter, mix, detect. The antenna was never a different kind of object.

In [11]:
def draw_receiver(k, E_mV, f_mhz, RL, XL, detect_on):
    lam = C_LIGHT / (f_mhz * 1e6)
    Ra, Xa = 73.1, 42.5
    G, E = 1.64, E_mV * 1e-3
    heff = lam / np.pi
    Voc = E * heff
    Zl = complex(RL, XL)
    I = Voc / (complex(Ra, Xa) + Zl)
    Pl = 0.5 * abs(I) ** 2 * RL
    Pmax = Voc ** 2 / (8 * Ra)
    S = E ** 2 / (2 * ETA0)
    Ae = G * lam ** 2 / (4 * np.pi)
    Pem = S * Ae
    wt = 2 * np.pi * k / 120
    vmax = max(Voc, 1e-12)

    fig = plt.figure(figsize=(13.0, 5.2))
    gs = fig.add_gridspec(2, 3, width_ratios=[1.25, 1.2, 0.55],
                          wspace=0.28, hspace=0.46, left=0.02, right=0.995,
                          top=0.90, bottom=0.10)

    a0 = fig.add_subplot(gs[0, 0])
    sch_axes(a0, (-0.6, 4.8), (-1.6, 1.8))
    zz = np.linspace(-1.0, 1.0, 60)
    a0.plot(np.zeros_like(zz) + 0.5, zz, color=GREEN, lw=4)
    a0.plot(0.5 + dipole_current(zz / 4, 0.5) * np.cos(wt) * 0.35, zz,
            color=DOT, lw=1.4)
    wire(a0, [(0.5, 0.12), (1.6, 0.35)], Voc / 2, vmax)
    wire(a0, [(0.5, -0.12), (1.6, -0.35)], -Voc / 2, vmax)
    resistor(a0, (1.6, 0.35), (3.0, 0.35), Voc / 2, vmax, f"RL {RL:.0f}Ω")
    if XL >= 0:
        inductor(a0, (3.0, 0.35), (4.0, 0.35), 0.0, vmax, f"+j{XL:.0f}")
    else:
        capacitor(a0, (3.0, 0.35), (4.0, 0.35), 0.0, vmax, f"−j{abs(XL):.0f}")
    wire(a0, [(4.0, 0.35), (4.4, 0.35), (4.4, -0.35), (1.6, -0.35)],
         -Voc / 2, vmax)
    charge_dots(a0, seg((1.6, 0.35), (3.0, 0.35), 30), k / 120 * abs(I) * 3e5,
                spacing=0.24, ms=3.4)
    a0.set_title("the wave drives the wire, the wire drives the load",
                 fontsize=8.5)

    a1 = fig.add_subplot(gs[1, 0])
    sch_axes(a1, (-0.6, 4.8), (-1.0, 1.4))
    source(a1, (0.6, -0.5), (0.6, 0.7), 0.0, vmax, "ac", f"Voc {Voc*1e3:.3f}mV")
    resistor(a1, (0.6, 0.7), (1.9, 0.7), Voc / 2, vmax, f"Ra {Ra:.0f}Ω")
    inductor(a1, (1.9, 0.7), (2.9, 0.7), Voc / 3, vmax, f"Xa +{Xa:.0f}")
    resistor(a1, (2.9, 0.7), (4.2, 0.7), Voc / 4, vmax, f"load")
    wire(a1, [(4.2, 0.7), (4.5, 0.7), (4.5, -0.5), (0.6, -0.5)], 0.0, vmax)
    charge_dots(a1, loop_rect(0.6, 4.5, -0.5, 0.7), k / 120 * abs(I) * 3e5,
                spacing=0.24, ms=3.4)
    a1.set_title("the same thing as a Thévenin source", fontsize=8.5)

    a2 = panel(fig.add_subplot(gs[0, 1]), BLUE)
    rr = np.logspace(0, 4, 400)
    P = np.array([0.5 * abs(Voc / (complex(Ra, Xa) + complex(r, XL))) ** 2 * r
                  for r in rr])
    a2.semilogx(rr, P * 1e12, color=POS, lw=1.7)
    a2.axvline(Ra, color=DOT, lw=1.1, ls="--")
    a2.text(Ra * 1.2, P.max() * 1e12 * 0.5, "RL = Ra", color=DOT, fontsize=7.5)
    a2.plot([RL], [Pl * 1e12], "o", ms=8, color=ORANGE)
    a2.set_xlabel("load resistance  (Ω)"); a2.set_ylabel("power  (pW)")
    a2.set_title(f"peak {Pmax*1e12:.4f} pW at the conjugate match")

    a3 = panel(fig.add_subplot(gs[1, 1]), GREEN)
    a3.bar([0, 1], [Pmax * 1e12, Pem * 1e12], color=[POS, PURP], width=0.5)
    for x, v in ((0, Pmax * 1e12), (1, Pem * 1e12)):
        a3.text(x, v, f"  {v:.5f} pW", ha="center", va="bottom", fontsize=8)
    a3.set_xticks([0, 1])
    a3.set_xticklabels(["circuit:\n$V_{oc}^2/8R_a$", "field:\n$S\\cdot A_e$"],
                       fontsize=8)
    a3.set_ylabel("available power  (pW)")
    a3.set_ylim(0, max(Pmax, Pem) * 1e12 * 1.35)
    a3.set_title(f"two independent routes — agree to "
                 f"{abs(Pmax-Pem)/Pem*100:.3f}%")

    readout(fig, 0.845, 0.90, [
        "INCOMING", "─" * 26,
        f"|E|         {E_mV:>10.3f}mV/m",
        f"frequency   {f_mhz:>10.1f}MHz",
        f"λ           {lam:>10.3f}m",
        f"S = E²/2η   {S*1e9:>10.4f}nW/m²",
        "", "ANTENNA", "─" * 26,
        f"h_eff = λ/π {heff:>10.4f}m",
        f"Voc         {Voc*1e3:>10.5f}mV",
        f"Ra          {Ra:>10.1f}Ω",
        f"Xa          {Xa:>+10.1f}Ω",
        f"Ae = Gλ²/4π {Ae:>10.5f}m²",
        "", "LOAD", "─" * 26,
        f"RL          {RL:>10.1f}Ω",
        f"XL          {XL:>+10.1f}Ω",
        f"conjugate?  {str(abs(RL-Ra)<1 and abs(XL+Xa)<1):>14s}",
        f"|I|         {abs(I)*1e6:>10.4f}µA",
        f"P delivered {Pl*1e12:>10.5f}pW",
        "", "AVAILABLE", "─" * 26,
        f"Voc²/8Ra    {Pmax*1e12:>10.5f}pW",
        f"S·Ae        {Pem*1e12:>10.5f}pW",
        f"difference  {abs(Pmax-Pem)/Pem*100:>10.3f}%",
    ], color=GREEN if abs(RL - Ra) < 1 and abs(XL + Xa) < 1 else FG)
    footer(fig, f"Voc = E·λ/π   ·   P = Voc²/8Ra = S·Ae   ·   "
                f"circuit theory and field theory give the same number")
    plt.show()


_pD, _sD = timeline(119, step=2)
wD = dict(E_mV=widgets.FloatSlider(value=1.0, min=0.1, max=10, step=0.1,
                                   description="|E| (mV/m):", **SL),
          f_mhz=widgets.FloatSlider(value=100, min=50, max=300, step=10,
                                    description="frequency MHz:", **SL),
          RL=widgets.FloatSlider(value=73, min=5, max=600, step=1,
                                 description="load R (Ω):", **SL),
          XL=widgets.FloatSlider(value=-42.5, min=-200, max=200, step=2.5,
                                 description="load X (Ω):", **SL),
          detect_on=widgets.Checkbox(value=False, description="(reserved)",
                                     indent=False),
          k=_sD)
display(widgets.VBox([widgets.HBox([wD["E_mV"], wD["f_mhz"], wD["RL"]]),
                      widgets.HBox([wD["XL"], _pD, _sD])]),
        widgets.interactive_output(draw_receiver, wD))

Output()